In [1]:
import utils
import numpy as np
import pandas as pd
import os
from scipy.stats import chisquare

import matplotlib.pyplot as plt
import seaborn as sns # Loads in the theme
sns.set_theme()

pd.set_option('future.no_silent_downcasting', True)
pd.set_option('display.max_rows', None)

In [2]:
# Make output folders
output_dir = os.path.join('..', '..','pre-processed', 'modelling_data')
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

In [3]:
# Preparation
experiment_output_path = os.path.join('..', '..', '..', 'phd_1_task', 'experiment_output', 'behavioural_files')
#experiment_output_path = os.path.join('..', '..', '..', 'phd_1_task', 'experiment_output_verena')

In [4]:
unique_subject_ids = list(range(902, 950))
unique_sessions = [3]

recent_results = utils.analysis_functions.list_files_with_date_and_subject_id(experiment_output_path, unique_subject_ids, unique_sessions)
print(recent_results)

sub-903 did not complete session-03
sub-904 did not complete session-03
sub-905 did not complete session-03
sub-906 did not complete session-03
sub-907 did not complete session-03
sub-908 did not complete session-03
sub-909 did not complete session-03
sub-910 did not complete session-03
sub-911 did not complete session-03
sub-912 did not complete session-03
sub-913 did not complete session-03
sub-914 did not complete session-03
sub-915 did not complete session-03
sub-916 did not complete session-03
sub-917 did not complete session-03
sub-918 did not complete session-03
sub-919 did not complete session-03
sub-920 did not complete session-03
sub-921 did not complete session-03
sub-922 did not complete session-03
sub-923 did not complete session-03
sub-924 did not complete session-03
sub-925 did not complete session-03
sub-926 did not complete session-03
sub-927 did not complete session-03
sub-928 did not complete session-03
sub-929 did not complete session-03
sub-930 did not complete ses

In [5]:
# Opens all the files and combines them into one dataframe
combined_results_dataframe = utils.analysis_functions.combine_result_files(recent_results)
if 'verena' in experiment_output_path:
    combined_dataframe = combined_results_dataframe[~combined_results_dataframe['stimuli_type'].isin([2, 4])]
    
print(combined_results_dataframe)

     trial  emotional_cue_time response objectively_correct  \
0        0        1.720101e+09       up                Even   
1        1        1.720101e+09     down                Even   
2        2        1.720101e+09       up                Even   
3        3        1.720101e+09       up                Even   
4        4        1.720101e+09     down                Even   
5        5        1.720101e+09     down                Even   
6        6        1.720101e+09       up                Even   
7        7        1.720101e+09     down                Even   
8        8        1.720101e+09       up                True   
9        9        1.720101e+09       up               False   
10      10        1.720101e+09     down                True   
11      11        1.720101e+09     down               False   
12      12        1.720101e+09       up                True   
13      13        1.720101e+09     down                True   
14      14        1.720101e+09     down                

# Check congruency, valence and volatility

In [6]:
# Check a block's congruency
combined_results_dataframe['congruency'] = combined_results_dataframe.apply(utils.analysis_functions.check_congruency, args=(True,), axis=1)

# Check a block's valence
combined_results_dataframe['valence'] = combined_results_dataframe.apply(utils.analysis_functions.check_valence, args=(True,), axis=1)

# Check a block's volatility
combined_results_dataframe = combined_results_dataframe[combined_results_dataframe['probability_condition'] != 50]
combined_results_dataframe = combined_results_dataframe.groupby('stimuli_type', group_keys=False).apply(lambda x: utils.analysis_functions.check_volatility(x, True))
combined_results_dataframe = combined_results_dataframe.sort_values(by=['stimuli_type','trial'], ascending=[True, True])
print(combined_results_dataframe[['volatility','probability_condition']])
combined_results_dataframe = combined_results_dataframe.sort_values(by=['subject_id', 'session', 'stimuli_type', 'trial'], ascending=[True, True, True, True])
combined_results_dataframe.reset_index(drop=True, inplace=True)
print(combined_results_dataframe[['subject_id', 'session', 'congruency', 'valence', 'temp_volatility', 'volatility']])
combined_results_dataframe = combined_results_dataframe.drop(columns=['temp_volatility'])

print(combined_results_dataframe['valence'].unique())
print(combined_results_dataframe['congruency'].unique())
print(combined_results_dataframe['volatility'].unique())

    volatility  probability_condition
0            0                     80
1            0                     80
2            0                     80
3            0                     80
4            0                     80
5            0                     80
6            0                     80
7            0                     80
8            0                     80
9            0                     80
10           0                     80
11           0                     80
12           0                     80
13           0                     80
14           0                     80
15           0                     80
16           0                     80
17           0                     80
18           0                     80
19           0                     80
20           0                     80
21           0                     80
22           0                     80
23           0                     80
24           0                     80
25          

/var/folders/bh/ld02p1_550d75y0x32dmpb540000gp/T/ipykernel_6344/2193021109.py:9: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  combined_results_dataframe = combined_results_dataframe.groupby('stimuli_type', group_keys=False).apply(lambda x: utils.analysis_functions.check_volatility(x, True))


# Recode existing variables

In [7]:
combined_results_dataframe = combined_results_dataframe.dropna(subset=['response'])

combined_results_dataframe = utils.analysis_functions.replace_strings_with_integers(combined_results_dataframe,
                                                                                    {'up': 2, 'down': 1},
                                                                                    'response',
                                                                                    'response_int')

combined_results_dataframe = utils.analysis_functions.replace_strings_with_integers(combined_results_dataframe,
                                                                                    {3: 1, 1: 2},
                                                                                    'stimuli_type',
                                                                                    'stimuli_type')

combined_results_dataframe = utils.analysis_functions.replace_strings_with_integers(combined_results_dataframe,
                                                                                    {'True': 1, 'False': 0},
                                                                                    'subjectively_correct',
                                                                                    'subjectively_correct_int')

combined_results_dataframe = utils.analysis_functions.replace_strings_with_integers(combined_results_dataframe,
                                                                                    {'True': 1, 'False': 0},
                                                                                    'objectively_correct',
                                                                                    'objectively_correct_int')

combined_results_dataframe['subject_id'] = combined_results_dataframe['subject_id'].str.replace(r'\D', '', regex=True)

combined_results_dataframe['session'] = combined_results_dataframe['session'].str.replace(r'\D', '', regex=True)

# Remove everything but x trials after response

In [8]:
'''
for index_stimuli in unique_stimuli_types:
    for index_subject_id in unique_subject_ids:
        dataframe_single_group = input_dataframe[(input_dataframe['stimuli_type'] == index_stimuli) &
                                                 (input_dataframe['subject_id'] == index_subject_id)]

        # Add column that notes if a switch occurred recently
        dataframe_single_group['switch'] = dataframe_single_group['probability_condition'].diff().ne(0)
        # Filter out trials with 50% reward probability
        dataframe_for_switches = dataframe_single_group  #.reset_index()
        dataframe_for_switches = dataframe_for_switches.drop(dataframe_for_switches
                                                             [(
                                                                      dataframe_for_switches.probability_condition == 50) |
                                                              (
                                                                      dataframe_for_switches.objectively_correct == 'Late')].index,
                                                             inplace=False)
'''

"\nfor index_stimuli in unique_stimuli_types:\n    for index_subject_id in unique_subject_ids:\n        dataframe_single_group = input_dataframe[(input_dataframe['stimuli_type'] == index_stimuli) &\n                                                 (input_dataframe['subject_id'] == index_subject_id)]\n\n        # Add column that notes if a switch occurred recently\n        dataframe_single_group['switch'] = dataframe_single_group['probability_condition'].diff().ne(0)\n        # Filter out trials with 50% reward probability\n        dataframe_for_switches = dataframe_single_group  #.reset_index()\n        dataframe_for_switches = dataframe_for_switches.drop(dataframe_for_switches\n                                                             [(\n                                                                      dataframe_for_switches.probability_condition == 50) |\n                                                              (\n                                                         

# Now export the data

In [9]:
combined_results_dataframe.to_csv(f'{output_dir}/combined_dataframe_binary.csv', sep=';', index_label='index')